# Fase 4c — LSTM Clássico (contraparte do QLSTM)

Contraparte clássica do Quantum LSTM: mesma célula recorrente, mas com as quatro portas realizadas por transformações lineares neurais (em vez de circuitos variacionais). Mesmo protocolo do QLSTM — janela W=4, estado oculto de dimensão 4, otimizador Adam(0.05), 30 épocas, 3 réplicas bootstrap, alvo em log1p — para isolar a contribuição do circuito quântico. Kernel: `qml_dengue`.

In [1]:
import warnings; warnings.filterwarnings("ignore")
import os, json, sys, time
import numpy as np
import matplotlib.pyplot as plt
import pennylane as qml
from pennylane import numpy as pnp

REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
sys.path.insert(0, os.path.join(REPO_ROOT, "src"))
sys.path.insert(0, os.path.abspath(".."))
from feature_engineering import construir_features, splits_validacao
from utils_qml import metricas, salvar_padrao, plot_pred, validar_json_saida

CACHE = os.path.join(REPO_ROOT, "data", "dados_dengue_df_real.json")
with open(CACHE, encoding="utf-8") as f:
    dados_brutos = json.load(f)
dataset = construir_features(dados_brutos, n_lags=4)
splits  = splits_validacao(dataset)
NF = len(dataset["feature_names"])

NOMES = {0: ("C1", "Transmissão normal/crescente  (out/2023 – out/2024)"),
         1: ("C2", "Pico recorde 25.714 casos/sem  (jun/2024 – jun/2025)"),
         2: ("C3", "Pós-surto, Rt < 1              (out/2024 – jun/2025)")}
CENARIOS = {}
for idx, split in enumerate(splits[:3]):
    nome, desc = NOMES[idx]
    tr, te = split["treino"], split["teste"]
    CENARIOS[nome] = {"X_train": np.array(tr["X"]), "y_train": np.array(tr["y"]),
                      "X_test": np.array(te["X"]), "y_test": np.array(te["y"]), "nome": desc}
print(f"Features: {NF}")
for c, d in CENARIOS.items():
    print(f"  {c}: treino={len(d['X_train'])} | teste={len(d['X_test'])} | pico_teste={d['y_test'].max():.0f}")

Features: 13
  C1: treino=36 | teste=143 | pico_teste=25714
  C2: treino=88 | teste=91 | pico_teste=25714
  C3: treino=125 | teste=54 | pico_teste=947


In [2]:
# ── LSTM clássico: célula recorrente com 4 portas lineares (neurais) ──
W = 4    # comprimento da sequência
H = 4    # dimensão do estado oculto (mesma do QLSTM = 4 qubits)

def sig(z): return 1.0 / (1.0 + pnp.exp(-z))

def init_params(seed=0):
    rng = np.random.RandomState(seed); p = {}
    Din = H + NF                       # entrada das portas: [h ; x_t]
    esc = 1.0 / np.sqrt(Din)
    for g in ["f", "i", "g", "o"]:
        p["W"+g] = pnp.array(rng.normal(0, esc, (H, Din)), requires_grad=True)
        p["b"+g] = pnp.array(np.zeros(H), requires_grad=True)
    p["Wout"] = pnp.array(rng.normal(0, 0.3, (H,)), requires_grad=True)
    p["bout"] = pnp.array(0.0, requires_grad=True)
    return p

def forward_seq(seq, p):
    h = pnp.zeros(H); c = pnp.zeros(H)
    for t in range(seq.shape[0]):
        v = pnp.concatenate([h, seq[t]])          # [h ; x_t]
        f = sig(p["Wf"] @ v + p["bf"])
        i = sig(p["Wi"] @ v + p["bi"])
        g = pnp.tanh(p["Wg"] @ v + p["bg"])
        o = sig(p["Wo"] @ v + p["bo"])
        c = f * c + i * g
        h = o * pnp.tanh(c)
    return p["Wout"] @ h + p["bout"]

print(f"LSTM clássico definido: estado oculto H={H} | janela W={W}")

LSTM clássico definido: estado oculto H=4 | janela W=4


In [3]:
# ── Sequências e treino (mesmo protocolo do QLSTM) ──
def build_seqs_train(Xn, y):
    seqs = np.array([Xn[i-W+1:i+1] for i in range(W-1, len(Xn))])
    return seqs, y[W-1:]

def build_seqs_test(Xtr_n, Xte_n):
    ext = np.vstack([Xtr_n[-(W-1):], Xte_n]) if W > 1 else Xte_n
    return np.array([ext[i-W+1:i+1] for i in range(W-1, len(ext))])

def predict(seqs, p):
    return np.array([float(forward_seq(pnp.array(s), p)) for s in seqs])

def cost(p, Xb, yb):
    preds = pnp.stack([forward_seq(pnp.array(s), p) for s in Xb])
    return pnp.mean((preds - yb) ** 2)

def train_one(seqs, ylog_n, seed, epochs):
    p = init_params(seed); rng = np.random.RandomState(seed)
    idx = rng.choice(len(seqs), len(seqs), replace=True)   # bootstrap
    Xb, yb = seqs[idx], ylog_n[idx]
    opt = qml.AdamOptimizer(0.05)
    for _ in range(epochs):
        p, _ = opt.step_and_cost(lambda pp: cost(pp, Xb, yb), p)
    return p

print("Funções de sequência e treino prontas.")

Funções de sequência e treino prontas.


In [4]:
# ── Treino nos 3 cenários (rápido — sem circuitos quânticos) ──
B  = 3
EP = 30
np.random.seed(42)
RESULTADOS = {}

for cen, d in CENARIOS.items():
    t0 = time.time()
    Xtr, ytr = d["X_train"], d["y_train"]
    Xte, yte = d["X_test"],  d["y_test"]
    mu, sd = Xtr.mean(0), Xtr.std(0) + 1e-6
    Xtr_n, Xte_n = (Xtr - mu) / sd, (Xte - mu) / sd
    seqs_tr, ys_tr = build_seqs_train(Xtr_n, ytr)
    seqs_te = build_seqs_test(Xtr_n, Xte_n)
    ylog_mu, ylog_sd = np.log1p(ys_tr).mean(), np.log1p(ys_tr).std() + 1e-6
    ylog_n = (np.log1p(ys_tr) - ylog_mu) / ylog_sd
    def denorm(pn): return np.maximum(np.expm1(pn * ylog_sd + ylog_mu), 0)
    preds = np.zeros((B, len(seqs_te)))
    for b in range(B):
        p = train_one(seqs_tr, ylog_n, seed=42 + b, epochs=EP)
        preds[b] = denorm(predict(seqs_te, p))
    med = np.median(preds, axis=0)
    m = metricas(yte, med, preds, nome=f"LSTM_{cen}")
    RESULTADOS[cen] = {**m, "preds_matrix": preds, "mediana": med,
                       "y_test": yte, "tempo_s": time.time() - t0}
    print(f"{cen}: R2={m['R2']:.4f} | RMSE={m['RMSE']:.1f} | WIS={m['WIS']:.2f} | {RESULTADOS[cen]['tempo_s']:.0f}s")

C1: R2=-0.0565 | RMSE=5882.4 | WIS=2581.93 | 3s
C2: R2=-0.1689 | RMSE=7500.3 | WIS=3544.24 | 7s
C3: R2=-34.3787 | RMSE=1124.3 | WIS=1068.10 | 9s


In [5]:
# ── Tabela + figura previsto vs. observado ──
print(f"\n{'='*60}")
print(f"{'FASE 4 — LSTM Clássico (contraparte do QLSTM)':^60}")
print(f"{'='*60}")
print(f"{'Cenário':<10}{'R²':>10}{'RMSE':>12}{'WIS':>12}")
print("-"*60)
for cen, r in RESULTADOS.items():
    print(f"{cen:<10}{r['R2']:>10.4f}{r['RMSE']:>12.1f}{r['WIS']:>12.2f}")
print("="*60)

plot_pred(RESULTADOS,
          "Fase 4 — LSTM Clássico: Predição vs. Observado (dados reais DF 2022-2025)",
          "fase4_lstm_pred_vs_obs.png")


       FASE 4 — LSTM Clássico (contraparte do QLSTM)        
Cenário           R²        RMSE         WIS
------------------------------------------------------------
C1           -0.0565      5882.4     2581.93
C2           -0.1689      7500.3     3544.24
C3          -34.3787      1124.3     1068.10
[SALVO] fase4_lstm_pred_vs_obs.png


In [6]:
SCHEMA_INFO = {
    "algoritmo": "LSTM_Classico",
    "fase": 4,
    "tipo": "classico_recorrente",
    "n_parametros_quanticos": None,
    "config": {"H": H, "W": W, "n_epochs": EP, "n_bootstrap": B,
               "otimizador": "Adam(0.05)", "alvo": "log1p"},
}
doc = salvar_padrao(RESULTADOS, SCHEMA_INFO)
validar_json_saida(doc, contexto="LSTM_Classico")

[PADRAO] fase04_lstm_classico_resultados.json
  Algoritmo : LSTM_Classico
  Tipo      : classico_recorrente
  Parametros quanticos: None
  C1: R2=-0.0565 | WIS=2581.93 | 2.6s
  C2: R2=-0.1689 | WIS=3544.24 | 7.3s
  C3: R2=-34.3787 | WIS=1068.10 | 9.2s
[CONTRATO OK] [LSTM_Classico] JSON valido — todos os campos e invariantes corretos


True